# 기존 애니메이션 외부 ID 매핑

DB의 기존 anime (Laftel 크롤링 데이터)를 AniList / MyAnimeList와 매핑합니다.

**매핑 전략 (하이브리드)**
1. **Phase 1**: anime-offline-database (오프라인 JSON)에서 제목/synonyms 정확 매칭 → MAL ID + AniList ID 확보
2. **Phase 2**: Phase 1 미매칭분 → AniList API 검색 → 제목+연도 매칭
3. 결과 병합 후 `anime_external_id` 테이블에 저장

In [9]:
import sys
import time
import re
import json
import dotenv
import os
import psycopg2
from psycopg2.extras import execute_batch

sys.path.insert(0, "scripts")
from external_api import (
    anilist_search,
    match_anilist_by_title_and_year,
)

dotenv.load_dotenv(dotenv.find_dotenv())

pg = psycopg2.connect(
    host=os.environ["POSTGRES_HOST"],
    port=os.environ["POSTGRES_PORT"],
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
)
cur = pg.cursor()

print("DB 연결 완료")

DB 연결 완료


In [2]:
# anime_external_id 테이블 생성/마이그레이션
cur.execute("""
    CREATE TABLE IF NOT EXISTS anime_external_id (
        anime_id       BIGINT PRIMARY KEY REFERENCES anime(id),
        mal_id         INTEGER,
        anilist_id     INTEGER,
        mal_title      VARCHAR(500),
        anilist_title  VARCHAR(500),
        matched_at     TIMESTAMP NOT NULL DEFAULT NOW(),
        UNIQUE(mal_id),
        UNIQUE(anilist_id)
    );
""")

# 기존 테이블에 aniplus 컬럼이 있으면 마이그레이션
cur.execute("""
    DO $$
    BEGIN
        IF EXISTS (SELECT 1 FROM information_schema.columns
                   WHERE table_name = 'anime_external_id' AND column_name = 'aniplus_id') THEN
            ALTER TABLE anime_external_id DROP COLUMN aniplus_id;
            ALTER TABLE anime_external_id DROP COLUMN aniplus_title_ko;
        END IF;
        IF NOT EXISTS (SELECT 1 FROM information_schema.columns
                       WHERE table_name = 'anime_external_id' AND column_name = 'anilist_id') THEN
            ALTER TABLE anime_external_id ADD COLUMN anilist_id INTEGER;
            ALTER TABLE anime_external_id ADD COLUMN anilist_title VARCHAR(500);
        END IF;
    END $$;
""")

pg.commit()
print("테이블 준비 완료")

테이블 준비 완료


In [3]:
# 매핑 안 된 anime 목록 조회
cur.execute("""
    SELECT a.id, a.name, a.air_year_quarter
    FROM anime a
    LEFT JOIN anime_external_id e ON a.id = e.anime_id
    WHERE e.anime_id IS NULL
    ORDER BY a.id
""")
unmapped = cur.fetchall()
print(f"매핑 필요한 anime: {len(unmapped)}개")

# 이미 매핑된 수
cur.execute("SELECT COUNT(*) FROM anime_external_id")
mapped_count = cur.fetchone()[0]
print(f"이미 매핑된 anime: {mapped_count}개")

매핑 필요한 anime: 8873개
이미 매핑된 anime: 0개


## Phase 1: anime-offline-database 오프라인 매칭

`anime-offline-database` JSON에서 제목/synonyms로 정확 매칭.
sources URL에서 MAL ID, AniList ID를 추출합니다.

다운로드: https://github.com/manami-project/anime-offline-database/releases

In [4]:
# anime-offline-database 로드 + 제목 인덱스 구축
OFFLINE_DB_PATH = "scripts/anime-db.json"

with open(OFFLINE_DB_PATH, encoding="utf-8") as f:
    offline_db = json.load(f)

print(f"오프라인 DB 로드: {len(offline_db['data'])}개 엔트리")

def extract_ids_from_sources(sources: list[str]) -> dict:
    """sources URL 배열에서 MAL ID, AniList ID 추출."""
    ids = {"mal_id": None, "anilist_id": None}
    for url in sources:
        m = re.search(r"myanimelist\.net/anime/(\d+)", url)
        if m:
            ids["mal_id"] = int(m.group(1))
        m = re.search(r"anilist\.co/anime/(\d+)", url)
        if m:
            ids["anilist_id"] = int(m.group(1))
    return ids

# 제목 → 엔트리 인덱스 (소문자 정규화)
# 하나의 제목에 여러 엔트리가 있을 수 있으므로 list로 저장
title_index: dict[str, list[dict]] = {}

for entry in offline_db["data"]:
    all_titles = [entry["title"]] + entry.get("synonyms", [])
    ids = extract_ids_from_sources(entry.get("sources", []))
    entry_info = {
        **ids,
        "title": entry["title"],
        "year": entry.get("animeSeason", {}).get("year"),
        "type": entry.get("type"),
    }
    for t in all_titles:
        key = t.strip().lower()
        if key:
            if key not in title_index:
                title_index[key] = []
            title_index[key].append(entry_info)

print(f"제목 인덱스: {len(title_index)}개 고유 제목")

# 한국어 제목 커버리지 확인
hangul_re = re.compile(r"[\uac00-\ud7af]")
korean_keys = sum(1 for k in title_index if hangul_re.search(k))
print(f"한국어 제목 수: {korean_keys}개")

오프라인 DB 로드: 40729개 엔트리
제목 인덱스: 249647개 고유 제목
한국어 제목 수: 4302개


In [5]:
def parse_year_from_air_yq(air_yq: str | None) -> int | None:
    """'2023년 1분기' → 2023"""
    if not air_yq:
        return None
    m = re.search(r"(\d{4})", air_yq)
    return int(m.group(1)) if m else None

# Phase 1: 오프라인 DB 제목 매칭
phase1_mapped = {}  # anime_id → {mal_id, anilist_id, anilist_title}
phase1_failed = []  # (anime_id, name, air_yq) 매핑 실패

for anime_id, name, air_yq in unmapped:
    key = name.strip().lower()
    candidates = title_index.get(key, [])

    if not candidates:
        phase1_failed.append((anime_id, name, air_yq))
        continue

    # 후보가 1개면 바로 매칭
    if len(candidates) == 1:
        best = candidates[0]
    else:
        # 연도로 필터링
        year = parse_year_from_air_yq(air_yq)
        best = None
        if year:
            for c in candidates:
                if c.get("year") == year:
                    best = c
                    break
        if not best:
            best = candidates[0]

    phase1_mapped[anime_id] = {
        "mal_id": best["mal_id"],
        "anilist_id": best["anilist_id"],
        "anilist_title": best["title"],
    }

print(f"Phase 1 (오프라인 DB) 매칭 완료")
print(f"  성공: {len(phase1_mapped)}개")
print(f"  실패: {len(phase1_failed)}개")
print(f"  매칭률: {len(phase1_mapped)/len(unmapped)*100:.1f}%")

Phase 1 (오프라인 DB) 매칭 완료
  성공: 1297개
  실패: 7576개
  매칭률: 14.6%


## Phase 2: AniList API 검색 (미매칭분)

Phase 1에서 매칭 실패한 anime에 대해 AniList API로 검색합니다.

In [6]:
# Phase 2: AniList API 검색 (Phase 1 미매칭분만)
phase2_mapped = {}  # anime_id → {mal_id, anilist_id, anilist_title}
phase2_failed = []  # (anime_id, name) 매핑 실패

print(f"Phase 2 대상: {len(phase1_failed)}개")

for i, (anime_id, name, air_yq) in enumerate(phase1_failed):
    if i > 0 and i % 100 == 0:
        print(f"  진행: {i}/{len(phase1_failed)} (매핑: {len(phase2_mapped)}, 실패: {len(phase2_failed)})")

    year = parse_year_from_air_yq(air_yq)

    try:
        candidates = anilist_search(name, per_page=10)
        best = match_anilist_by_title_and_year(candidates, name, year)

        if best:
            phase2_mapped[anime_id] = {
                "anilist_id": best["id"],
                "mal_id": best.get("idMal"),
                "anilist_title": best.get("title", {}).get("romaji", ""),
            }
        else:
            phase2_failed.append((anime_id, name))

    except Exception as e:
        phase2_failed.append((anime_id, name))
        if i < 5:
            print(f"  에러 [{anime_id}] {name}: {e}")

    time.sleep(0.7)  # AniList rate limit: 90 req/min

print(f"\nPhase 2 (AniList API) 매칭 완료")
print(f"  성공: {len(phase2_mapped)}개")
print(f"  실패: {len(phase2_failed)}개")

# 전체 결과 병합
all_mapped = {**phase1_mapped, **phase2_mapped}
all_failed = phase2_failed  # Phase 2까지 실패한 것만

print(f"\n=== 전체 매핑 결과 ===")
print(f"  Phase 1 (오프라인): {len(phase1_mapped)}개")
print(f"  Phase 2 (AniList):  {len(phase2_mapped)}개")
print(f"  합계: {len(all_mapped)}개 ({len(all_mapped)/len(unmapped)*100:.1f}%)")
print(f"  최종 미매칭: {len(all_failed)}개")

Phase 2 대상: 7576개
  진행: 100/7576 (매핑: 4, 실패: 96)
  진행: 200/7576 (매핑: 8, 실패: 192)
  진행: 300/7576 (매핑: 9, 실패: 291)
  진행: 400/7576 (매핑: 11, 실패: 389)
  진행: 500/7576 (매핑: 15, 실패: 485)
  진행: 600/7576 (매핑: 18, 실패: 582)
  진행: 700/7576 (매핑: 26, 실패: 674)
  진행: 800/7576 (매핑: 29, 실패: 771)
  진행: 900/7576 (매핑: 36, 실패: 864)
  진행: 1000/7576 (매핑: 38, 실패: 962)
  진행: 1100/7576 (매핑: 42, 실패: 1058)
  진행: 1200/7576 (매핑: 45, 실패: 1155)
  진행: 1300/7576 (매핑: 47, 실패: 1253)
  진행: 1400/7576 (매핑: 48, 실패: 1352)
  진행: 1500/7576 (매핑: 49, 실패: 1451)
  진행: 1600/7576 (매핑: 50, 실패: 1550)
  진행: 1700/7576 (매핑: 50, 실패: 1650)
  진행: 1800/7576 (매핑: 52, 실패: 1748)
  진행: 1900/7576 (매핑: 58, 실패: 1842)
  진행: 2000/7576 (매핑: 60, 실패: 1940)
  진행: 2100/7576 (매핑: 61, 실패: 2039)
  진행: 2200/7576 (매핑: 63, 실패: 2137)
  진행: 2300/7576 (매핑: 71, 실패: 2229)
  진행: 2400/7576 (매핑: 75, 실패: 2325)
  진행: 2500/7576 (매핑: 76, 실패: 2424)
  진행: 2600/7576 (매핑: 78, 실패: 2522)
  진행: 2700/7576 (매핑: 79, 실패: 2621)
  진행: 2800/7576 (매핑: 80, 실패: 2720)
  진행: 2900/7576 (매핑: 82, 

## 매핑 결과 DB 저장

In [14]:
# 매핑 결과를 anime_external_id에 저장
# mal_id, anilist_id 중복 방지: 동일 외부 ID에 여러 anime가 매핑되면 첫 번째만 유지
seen_mal_ids = set()
seen_anilist_ids = set()
insert_rows = []
dup_count = 0

for anime_id, info in all_mapped.items():
    mal_id = info["mal_id"]
    anilist_id = info["anilist_id"]

    # 중복 체크: mal_id 또는 anilist_id가 이미 사용됐으면 None으로 대체
    if mal_id and mal_id in seen_mal_ids:
        mal_id = None
        dup_count += 1
    if anilist_id and anilist_id in seen_anilist_ids:
        anilist_id = None

    if mal_id:
        seen_mal_ids.add(mal_id)
    if anilist_id:
        seen_anilist_ids.add(anilist_id)

    insert_rows.append((
        anime_id,
        mal_id,
        anilist_id,
        "",
        info["anilist_title"],
    ))

print(f"저장할 매핑 수: {len(insert_rows)}")
print(f"  MAL ID 있음: {sum(1 for r in insert_rows if r[1])}개")
print(f"  AniList ID 있음: {sum(1 for r in insert_rows if r[2])}개")
print(f"  중복 외부 ID 제거: {dup_count}개")

execute_batch(cur, """
    INSERT INTO anime_external_id (anime_id, mal_id, anilist_id, mal_title, anilist_title)
    VALUES (%s, %s, %s, %s, %s)
    ON CONFLICT (anime_id) DO UPDATE SET
        mal_id = EXCLUDED.mal_id,
        anilist_id = EXCLUDED.anilist_id,
        mal_title = EXCLUDED.mal_title,
        anilist_title = EXCLUDED.anilist_title,
        matched_at = NOW();
""", insert_rows, page_size=500)

pg.commit()
print(f"\n{len(insert_rows)}개 매핑 저장 완료")

저장할 매핑 수: 1461
  MAL ID 있음: 1303개
  AniList ID 있음: 1193개
  중복 외부 ID 제거: 61개

1461개 매핑 저장 완료


## 매핑 실패 리포트

In [15]:
# 매핑 통계
cur.execute("SELECT COUNT(*) FROM anime")
total_anime = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM anime_external_id")
total_mapped = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM anime_external_id WHERE mal_id IS NOT NULL")
mal_count = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM anime_external_id WHERE anilist_id IS NOT NULL")
anilist_count = cur.fetchone()[0]

print(f"=== 매핑 통계 ===")
print(f"전체 anime: {total_anime}")
print(f"매핑 완료: {total_mapped} ({total_mapped/total_anime*100:.1f}%)")
print(f"  AniList: {anilist_count}")
print(f"  MAL: {mal_count}")
print(f"미매핑: {total_anime - total_mapped}")

# 매핑 실패 샘플 (수동 검토용)
print(f"\n--- 최종 매핑 실패 샘플 (상위 30개) ---")
for anime_id, name in all_failed[:30]:
    print(f"  [{anime_id}] {name}")

=== 매핑 통계 ===
전체 anime: 8873
매핑 완료: 1461 (16.5%)
  AniList: 1193
  MAL: 1303
미매핑: 7412

--- 최종 매핑 실패 샘플 (상위 30개) ---
  [12912] 노기자카 하루카의 비밀
  [13162] 아기곰 미샤
  [13167] A 채널
  [13169] 쥬얼펫 선샤인
  [13171] 롯테의 장난감!
  [13172] 신만이 아는 세계 2기
  [13173] 노미오와 줄리엣
  [13183] 마리아 홀릭 얼라이브
  [13184] 역경무뢰 카이지 시즌 2 part 1
  [13186] 스켓 댄스
  [13187] 효우게모노
  [13188] 잃어버린 마법의 섬 홋타라케
  [13195] 도그 데이즈 1기
  [13197] 최유기 외전 OVA
  [13205] 라푼젤
  [13209] 극장판 닷핵퀀텀 : 숨겨진 몬스터의 비밀 .hack//Quantum
  [13212] 알파 앤 오메가
  [13213] 랭고
  [13217] 플래닛 51
  [13220] 리오 레인보우 게이트
  [13221] 쓰리몬 2기
  [13222] 카드파이트 뱅가드
  [13223] 다람쥐 구조대
  [13224] (더빙) 너에게 닿기를 2기 리마스터
  [13225] IS 인피니트 스트라토스
  [13230] 메가마인드
  [13231] 패트와 매트
  [13232] 탐정 오페라 밀키홈즈
  [13233] (더빙) 사무라이 잭
  [13234] 드래곤 크라이시스


In [16]:
cur.close()
pg.close()
print("Done!")

Done!


In [13]:
pg.rollback()